# STL-10 모델 성능 개선 보고서

이번 과제의 핵심은 단순히 모델 하나를 더 돌리는 게 아니라, **왜 이 방법을 썼고, 점수가 왜 올랐고, 다음에는 어디를 더 건드리면 되는지**를 보여주는 것이다.

현재 기준 최고 결과는 **ResNet18 fine-tuning + pseudo-label** 조합이고, test accuracy는 **0.9557**까지 확인했다. 여기서 더 올리려면 방향은 명확하다. 작은 직접 CNN을 더 키우는 것보다, **사전학습 모델을 더 강하게 쓰고, unlabeled 데이터를 더 조심스럽게 붙이고, 마지막 제출은 앙상블/TTA로 안정화**하는 쪽이 점수 상승 가능성이 크다.

## 1. 대회와 데이터 이해

STL-10은 10개 클래스를 분류하는 이미지 인식 문제다. 클래스는 airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck이다. 이미지는 RGB 96×96 크기이고, labeled train은 5,000장, test는 8,000장이다.

이 데이터셋에서 제일 중요한 특징은 **정답 없는 unlabeled 이미지가 100,000장 존재한다는 점**이다. 즉, 그냥 supervised CNN만 돌리면 labeled 5,000장만 보게 되지만, pseudo-label이나 semi-supervised 전략을 쓰면 unlabeled 데이터까지 학습에 끌어올 수 있다.

그래서 이번 개선 방향은 이렇게 잡았다.

| 방향 | 이유 |
|---|---|
| 강한 데이터 증강 | train 5,000장이 적어서 과적합을 막아야 함 |
| ImageNet 사전학습 모델 | STL-10이 ImageNet에서 가져온 이미지 기반이라 transfer learning 효과가 큼 |
| pseudo-label | unlabeled 100,000장을 일부라도 학습에 활용 가능 |
| TTA와 앙상블 | 단일 모델의 운을 줄이고 제출 점수를 안정화 |
| 제출 제한 관리 | 하루 제출 횟수를 validation 실험으로 아껴야 함 |

참고한 근거: STL-10 공식 설명은 unlabeled 데이터 활용을 주요 도전으로 제시하고, PyTorch STL10 문서도 train/test/unlabeled/train+unlabeled split을 지원한다.

In [ ]:
# 기본 설정
import os
import random
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset, Dataset, ConcatDataset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights, ResNet50_Weights, EfficientNet_B0_Weights

SEED = 40
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./data"
NUM_CLASSES = 10
BATCH_SIZE = 64
NUM_WORKERS = 2

class_names = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]

print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. 검증 데이터 구축 전략

이번 과제에서 제일 조심해야 하는 부분은 test set을 계속 보면서 모델을 고르는 행동이다. 그러면 실제 제출 점수에 과적합될 수 있다.

그래서 기준은 다음처럼 둔다.

1. train 5,000장을 8:2로 나눠서 **train 4,000장 / validation 1,000장**으로 사용한다.
2. 모든 실험은 같은 seed와 같은 validation index를 사용한다.
3. 모델 선택은 validation accuracy 기준으로 하고, test accuracy는 최종 확인용으로만 본다.
4. pseudo-label은 validation에 섞지 않는다. validation은 깨끗한 labeled 데이터로 남겨야 한다.

이렇게 해야 “진짜 좋아진 모델”인지, 아니면 test에 우연히 맞은 모델인지 구분할 수 있다.

In [ ]:
# 직접 CNN용 transform: 96x96 유지
custom_train_transform = transforms.Compose([
    transforms.RandomCrop(96, padding=12),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4467, 0.4398, 0.4066], std=[0.2603, 0.2566, 0.2713]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3))
])

custom_eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4467, 0.4398, 0.4066], std=[0.2603, 0.2566, 0.2713])
])

# 사전학습 모델용 transform: ImageNet 입력 기준 224x224
pretrained_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10))
])

pretrained_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 3. 데이터 처리 전략

처음부터 복잡하게 건드리기보다, 이미지 분류에서 바로 효과가 큰 처리부터 적용했다.

`RandomCrop`은 물체 위치가 조금 달라져도 맞히게 만든다. `HorizontalFlip`은 좌우 방향 변화에 강하게 만든다. `ColorJitter`는 밝기와 색감 차이에 덜 흔들리게 만든다. `RandomErasing`은 이미지 일부가 가려져도 전체 형태를 보고 맞히도록 만든다.

중요한 점은 validation/test에는 랜덤 증강을 넣지 않는 것이다. 평가할 때 이미지가 매번 바뀌면 모델 비교가 흔들리기 때문이다.

In [ ]:
# 데이터셋 로드
base_train = datasets.STL10(root=DATA_DIR, split="train", download=True, transform=custom_train_transform)
base_train_eval = datasets.STL10(root=DATA_DIR, split="train", download=False, transform=custom_eval_transform)
test_custom = datasets.STL10(root=DATA_DIR, split="test", download=True, transform=custom_eval_transform)

train_size = int(len(base_train) * 0.8)
val_size = len(base_train) - train_size
generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(base_train, [train_size, val_size], generator=generator)

train_indices = train_subset.indices
val_indices = val_subset.indices

train_loader_custom = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_custom = DataLoader(Subset(base_train_eval, val_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_custom = DataLoader(test_custom, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

pre_train = datasets.STL10(root=DATA_DIR, split="train", download=False, transform=pretrained_train_transform)
pre_train_eval = datasets.STL10(root=DATA_DIR, split="train", download=False, transform=pretrained_eval_transform)
test_pretrained = datasets.STL10(root=DATA_DIR, split="test", download=False, transform=pretrained_eval_transform)

train_loader_pre = DataLoader(Subset(pre_train, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_pre = DataLoader(Subset(pre_train_eval, val_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_pre = DataLoader(test_pretrained, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("train:", len(train_indices), "val:", len(val_indices), "test:", len(test_custom))

## 4. 학습 모델 개선 전략

이번 개선은 단계적으로 진행했다. 한 번에 강한 모델만 쓰면 “왜 점수가 올랐는지” 설명하기 어렵기 때문에, 직접 CNN → pseudo-label → pretrained model → pretrained + pseudo-label 순서로 비교했다.

| 실험 | 목적 | 핵심 특징 |
|---|---|---|
| Custom CNN + Augmentation | 직접 만든 CNN의 한계 확인 | 증강, BatchNorm, SiLU, Dropout |
| Custom CNN + Pseudo-label | unlabeled 데이터가 직접 CNN에도 도움이 되는지 확인 | confidence 0.95 이상만 사용 |
| ResNet18 Fine-tuning | 사전학습 모델 효과 확인 | ImageNet pretrained weight 사용 |
| ResNet18 + Pseudo-label | 현재 최고 조합 확인 | pretrained teacher로 pseudo-label 생성 |
| 다음 개선 후보 | 등수 상승용 | ResNet50, EfficientNet-B0, TTA, ensemble |

In [ ]:
# 공통 학습 / 평가 함수

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def fit_model(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4, patience=7):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history.append({"epoch": epoch, "train_acc": train_acc, "val_acc": val_acc, "val_loss": val_loss})
        print(f"[{epoch:02d}/{epochs}] train_acc={train_acc:.4f} val_acc={val_acc:.4f} val_loss={val_loss:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print("early stopping")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val_acc

In [ ]:
# 직접 구성 CNN
class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.SiLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.SiLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# 사전학습 모델 생성 함수

def build_resnet18(num_classes=10):
    weights = ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_resnet50(num_classes=10):
    weights = ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_efficientnet_b0(num_classes=10):
    weights = EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

## 5. 현재까지의 제출 점수 결과

이미 돌린 실험 기준으로 보면, 점수 상승 흐름이 꽤 깔끔하다.

| 제출/실험 | best validation accuracy | test accuracy | 해석 |
|---|---:|---:|---|
| Custom CNN + Augmentation | 0.8140 | 0.8026 | 직접 CNN은 좋아졌지만 데이터 수 한계가 있음 |
| Custom CNN + Pseudo-label | 0.8260 | 0.8099 | unlabeled를 붙였지만 teacher가 약해서 상승폭은 작음 |
| ResNet18 Fine-tuning | 0.9550 | 0.9536 | 사전학습 효과가 압도적으로 큼 |
| ResNet18 + Pseudo-label | 0.9600 | 0.9557 | 현재 최고, pseudo-label이 소폭 개선 |

결론은 명확하다. **이 과제에서 순위를 크게 올리는 핵심은 직접 CNN 구조 변경이 아니라 pretrained model + unlabeled 활용 + 앙상블이다.**

In [ ]:
# 결과 비교표
results = pd.DataFrame([
    {"experiment": "Custom CNN + Augmentation", "best_val_acc": 0.8140, "test_acc": 0.802625},
    {"experiment": "Custom CNN + Pseudo-label", "best_val_acc": 0.8260, "test_acc": 0.809875},
    {"experiment": "ResNet18 Fine-tuning", "best_val_acc": 0.9550, "test_acc": 0.953625},
    {"experiment": "ResNet18 + Pseudo-label", "best_val_acc": 0.9600, "test_acc": 0.955700},
])
results

In [ ]:
# 결과 시각화
plt.figure(figsize=(10, 4))
plt.bar(results["experiment"], results["test_acc"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("test accuracy")
plt.title("STL-10 Test Accuracy by Experiment")
plt.ylim(0.75, 1.0)
plt.show()

## 6. pseudo-label 활용 전략

pseudo-label은 정답 없는 이미지에 모델이 임시 정답을 붙이는 방식이다. 다만 전부 넣으면 위험하다. teacher 모델이 틀린 라벨을 많이 만들면, student 모델이 그 틀린 라벨까지 외워버린다.

그래서 기준을 이렇게 잡았다.

- confidence threshold: 0.95 이상
- validation 데이터는 절대 pseudo-label에 섞지 않음
- 직접 CNN teacher보다 ResNet18 teacher를 우선 사용
- 다음 개선에서는 threshold 0.90 / 0.95 / 0.98을 나눠서 validation 기준으로 비교

현재 결과를 보면 직접 CNN teacher는 2,161장을 선택했고 평균 confidence는 0.9818이었다. ResNet18 teacher는 1,768장을 선택했고 평균 confidence는 0.9622였다. 단순히 많이 고르는 것보다, **강한 teacher가 고른 데이터가 더 안전하다.**

In [ ]:
# pseudo-label 데이터셋 클래스
class PseudoLabelSTL10(Dataset):
    def __init__(self, root, indices, pseudo_labels, transform):
        self.dataset = datasets.STL10(root=root, split="unlabeled", download=False, transform=transform)
        self.indices = indices
        self.pseudo_labels = pseudo_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, _ = self.dataset[real_idx]
        y = int(self.pseudo_labels[idx])
        return x, y


@torch.no_grad()
def make_pseudo_labels(model, eval_transform, threshold=0.95, max_items=None, batch_size=128):
    model.eval()
    unlabeled_dataset = datasets.STL10(root=DATA_DIR, split="unlabeled", download=True, transform=eval_transform)

    if max_items is not None:
        unlabeled_dataset = Subset(unlabeled_dataset, list(range(max_items)))

    loader = DataLoader(unlabeled_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    selected_indices = []
    selected_labels = []
    selected_conf = []
    offset = 0

    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
        conf, pred = prob.max(dim=1)

        mask = conf >= threshold
        batch_indices = torch.arange(offset, offset + x.size(0))[mask.cpu()].tolist()

        selected_indices.extend(batch_indices)
        selected_labels.extend(pred[mask].cpu().tolist())
        selected_conf.extend(conf[mask].cpu().tolist())
        offset += x.size(0)

    print("selected pseudo samples:", len(selected_indices))
    if selected_conf:
        print("mean confidence:", float(np.mean(selected_conf)))

    return selected_indices, selected_labels

## 7. 학습 모델 선택 및 앙상블 전략

현재 ResNet18 하나만으로 0.9557까지 나왔기 때문에, 다음 제출에서 가장 효율적인 개선은 모델 종류를 늘리는 것이다.

추천 순서는 다음과 같다.

1. ResNet18 + pseudo-label: 현재 기준 모델
2. ResNet50 fine-tuning: 더 큰 backbone으로 성능 상승 기대
3. EfficientNet-B0 fine-tuning: 파라미터 대비 성능이 좋아서 앙상블 후보로 적합
4. 위 모델들의 soft-voting ensemble
5. 최종 제출 직전 TTA 적용

앙상블은 hard voting보다 soft voting이 낫다. hard voting은 각 모델의 최종 답만 보지만, soft voting은 “얼마나 확신했는지”까지 평균내기 때문이다.

In [ ]:
# TTA와 soft-voting 앙상블 평가 함수
# 핵심 아이디어: 원본 이미지와 좌우 반전 이미지를 모두 예측한 뒤 logits를 평균낸다.

@torch.no_grad()
def evaluate_tta_single(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits1 = model(x)
        logits2 = model(torch.flip(x, dims=[3]))
        logits = (logits1 + logits2) / 2

        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_ensemble_tta(models_list, loader, criterion, device):
    for m in models_list:
        m.eval()

    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits_sum = None
        for m in models_list:
            logits1 = m(x)
            logits2 = m(torch.flip(x, dims=[3]))
            logits = (logits1 + logits2) / 2
            logits_sum = logits if logits_sum is None else logits_sum + logits

        logits_avg = logits_sum / len(models_list)
        loss = criterion(logits_avg, y)

        total_loss += loss.item() * x.size(0)
        pred = logits_avg.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

## 8. 허용된 외부 데이터 검색/선택 및 활용 전략

이번 문제에서 외부 데이터를 무작정 추가하는 건 위험하다. 대회 규칙에서 허용되지 않으면 점수 자체가 무효가 될 수 있기 때문이다.

그래서 외부 데이터 활용은 세 단계로 나눈다.

| 우선순위 | 데이터/자원 | 사용 판단 |
|---|---|---|
| 1순위 | STL-10 공식 unlabeled 100,000장 | 데이터셋이 제공하는 공식 자원이라 가장 안전 |
| 2순위 | ImageNet pretrained weights | torchvision에서 제공하는 사전학습 가중치, 규칙 허용 시 사용 |
| 3순위 | 다른 이미지 데이터셋 추가 학습 | 규칙 확인 전에는 사용하지 않음 |

이번 노트북에서는 **공식 unlabeled + ImageNet pretrained weights**까지만 활용하는 방향이 제일 깔끔하다. 발표에서도 “규칙 위반 리스크가 적은 범위에서 성능을 끌어올렸다”고 말할 수 있다.

## 9. 제출 횟수 제한에 따른 제출 전략

하루 제출 횟수가 제한되어 있으면, public score를 실험용으로 쓰면 안 된다. 제출은 최종 확인용이어야 한다.

운영 방식은 이렇게 잡는다.

1. validation 기준으로만 후보 모델을 먼저 줄인다.
2. validation이 오른 모델만 제출 후보로 올린다.
3. 하루 첫 제출은 가장 안정적인 단일 모델로 한다.
4. 두 번째 제출부터는 pseudo-label threshold나 ensemble처럼 “변경 이유가 분명한 것”만 제출한다.
5. public score가 좋아도 validation이 나쁘면 최종 모델로 확정하지 않는다.

이번 과제 기준 제출 우선순위는 다음과 같다.

| 제출 순서 | 제출 내용 | 목적 |
|---:|---|---|
| 1 | ResNet18 + pseudo-label | 현재 최고 기준점 확보 |
| 2 | ResNet50 fine-tuning | backbone 확장 효과 확인 |
| 3 | EfficientNet-B0 fine-tuning | 다른 구조 모델 확보 |
| 4 | ResNet18 + ResNet50 ensemble | 앙상블 상승 확인 |
| 5 | Ensemble + TTA | 최종 안정화 제출 |

## 10. 제출 점수와 validation 점수 차이 원인 분석

현재 최고 모델은 validation accuracy 0.9600, test accuracy 0.9557이다. 차이는 약 0.0043 정도라서 큰 괴리는 아니다. 이 정도면 validation split이 test 난이도를 꽤 잘 반영했다고 볼 수 있다.

그래도 차이가 생긴 이유는 있다.

첫째, validation은 train 5,000장 중 1,000장이라 표본이 작다. test는 8,000장이므로 더 다양한 이미지가 들어온다.

둘째, STL-10의 unlabeled 데이터는 labeled 데이터와 비슷하지만 더 넓은 분포를 가진다. pseudo-label을 붙일 때 틀린 라벨이 일부 들어가면 test에서 약간 손해가 날 수 있다.

셋째, validation은 모델 선택에 반복적으로 사용된다. 그래서 validation에 약간 맞춰진 모델이 test에서는 조금 낮게 나올 수 있다.

결론적으로 현재 차이는 심각한 과적합 신호라기보다는, 작은 validation set과 pseudo-label 노이즈 때문에 생긴 자연스러운 차이에 가깝다.

## 11. 각 제출 별 개선 과정 정리

| 단계 | 바꾼 내용 | 왜 바꿨는지 | 결과 |
|---:|---|---|---:|
| 1 | 직접 CNN + 증강 | labeled 데이터 5,000장의 과적합 완화 | 0.8026 |
| 2 | 직접 CNN + pseudo-label | unlabeled 데이터 활용 가능성 확인 | 0.8099 |
| 3 | ResNet18 fine-tuning | ImageNet 특징을 STL-10에 전이 | 0.9536 |
| 4 | ResNet18 + pseudo-label | 강한 teacher로 unlabeled 일부 추가 | 0.9557 |
| 5 | ResNet50 / EfficientNet 후보 | backbone 다양화로 앙상블 준비 | 다음 실험 |
| 6 | TTA + soft ensemble | 최종 제출 점수 안정화 | 다음 실험 |

여기서 가장 큰 점프는 2번에서 3번으로 넘어갈 때다. 직접 CNN을 아무리 만지는 것보다 pretrained backbone을 쓰는 게 훨씬 컸다.

## 12. 목표 점수 설정

현재 최고 test accuracy가 0.9557이므로, 다음 목표는 현실적으로 두 단계로 잡는다.

| 목표 | 기준 |
|---|---:|
| 1차 목표 | 0.9600 이상 |
| 2차 목표 | 0.9650 이상 |
| 공격 목표 | 0.9700 근처 |

0.9600은 ResNet50이나 EfficientNet-B0 단일 모델만 좋아져도 가능성이 있다. 0.9650 이상은 단일 모델보다 **앙상블 + TTA + pseudo-label threshold 튜닝**까지 같이 들어가야 가능성이 크다.

## 13. 향후 모델 개선 계획

다음 개선은 무조건 점수 가능성이 큰 순서로 진행한다.

1. ResNet50 fine-tuning을 돌린다. ResNet18보다 표현력이 크기 때문에 validation이 오를 가능성이 높다.
2. EfficientNet-B0를 돌린다. ResNet 계열과 구조가 달라서 ensemble에 넣었을 때 서로 틀리는 부분이 다를 수 있다.
3. pseudo-label threshold를 0.90, 0.95, 0.98로 나눠 비교한다. 너무 낮으면 노이즈가 많고, 너무 높으면 데이터가 적다.
4. best model 2~3개를 soft voting한다.
5. 최종 제출에서 TTA를 적용한다.

내 판단으로는 이 과제에서 가장 효율적인 다음 한 방은 **ResNet18 + ResNet50 + EfficientNet-B0 soft ensemble**이다. 단일 모델 구조를 조금씩 뜯는 것보다 점수 상승 확률이 높다.

## 14. 모델 개선 과정에서 가장 어려웠던 점

가장 어려웠던 점은 pseudo-label을 어디까지 믿을지 정하는 부분이었다.

unlabeled 데이터는 100,000장이라 양은 많지만, 정답이 없다. 모델이 확신한다고 해서 항상 맞는 것도 아니다. 특히 STL-10 unlabeled에는 labeled 10개 클래스와 완전히 똑같지 않은 이미지도 들어갈 수 있다. 예를 들어 동물이나 차량 이미지는 있지만, 정확히 cat/dog/truck 같은 클래스와 애매하게 겹치는 경우가 생긴다.

그래서 pseudo-label은 많이 넣는 것보다 **깨끗하게 넣는 것**이 더 중요했다. 이 때문에 confidence threshold를 높게 잡고, validation 성능으로만 최종 판단했다.

## 15. AI로 코드를 만들 때 사용한 질문과 이유

| 질문 | 이 질문을 쓴 이유 |
|---|---|
| STL-10에서 labeled 데이터가 적고 unlabeled 데이터가 많은 상황에서 점수를 올리는 전략을 알려줘 | 문제 구조에 맞는 큰 방향을 잡기 위해 |
| PyTorch에서 STL-10용 ResNet18 fine-tuning 코드를 만들어줘 | 직접 CNN보다 pretrained 모델을 빠르게 실험하기 위해 |
| pseudo-label을 만들 때 confidence threshold를 적용하는 코드를 작성해줘 | unlabeled 데이터를 전부 넣지 않고 안전하게 고르기 위해 |
| validation set과 test set을 분리해서 모델을 선택하는 코드를 만들어줘 | 제출 점수에 과적합되는 것을 막기 위해 |
| 여러 모델의 logits를 평균내는 soft-voting ensemble 코드를 만들어줘 | 단일 모델보다 안정적인 최종 제출을 만들기 위해 |
| TTA로 원본 이미지와 좌우반전 이미지를 같이 예측하는 코드를 만들어줘 | 마지막 제출에서 작은 추가 상승을 노리기 위해 |

AI 질문은 그냥 “코드 짜줘”가 아니라, **내가 왜 이 방법을 쓰는지 먼저 정하고 그 부분만 구현시키는 방식**으로 사용했다. 그래야 코드가 과제 목적과 연결된다.

## 16. 발표용 핵심 흐름

10분 발표에서는 전부 설명하려고 하면 오히려 흐름이 흐려진다. 아래 순서로 말하면 된다.

1. STL-10은 train이 5,000장뿐이고 unlabeled가 100,000장이라, 적은 labeled 데이터를 어떻게 보완하느냐가 핵심이었다.
2. 처음에는 직접 CNN에 데이터 증강을 넣어 과적합을 줄였다. test accuracy는 0.8026이었다.
3. 그 다음 pseudo-label로 unlabeled 데이터를 일부 붙였고, 0.8099까지 올랐다. 다만 teacher가 약하면 상승폭이 크지 않았다.
4. 가장 큰 개선은 ResNet18 fine-tuning이었다. ImageNet에서 배운 특징을 가져오자 0.9536까지 올랐다.
5. 마지막으로 ResNet18 teacher가 확신하는 unlabeled 데이터만 붙여 0.9557까지 개선했다.
6. validation과 test 차이는 약 0.0043이라 심각한 과적합은 아니지만, pseudo-label 노이즈와 validation 표본 크기 때문에 차이가 생겼다고 봤다.
7. 다음 목표는 ResNet50, EfficientNet-B0, TTA, soft ensemble로 0.96 이상을 넘기는 것이다.

발표에서 제일 강조할 한 문장은 이것이다.

**“이번 과제에서 점수를 크게 올린 핵심은 CNN을 무작정 깊게 만든 것이 아니라, 데이터가 적은 문제 특성에 맞춰 pretrained model과 unlabeled data를 활용한 것이다.”**

## 17. 과제 관련 질문 및 추가 강의 요구 사항

이번 과제를 하면서 추가로 배우고 싶은 부분은 세 가지다.

첫째, pseudo-label threshold를 어떤 기준으로 정해야 하는지 더 알고 싶다. 0.95처럼 임의로 잡는 것 말고, validation 성능이나 class별 confidence 분포를 보고 정하는 방법을 배우고 싶다.

둘째, 앙상블을 할 때 단순 평균 말고 모델별 가중치를 어떻게 정하는지 궁금하다. validation 성능이 높은 모델에 더 큰 가중치를 주는 방식이 실제로 안정적인지도 확인하고 싶다.

셋째, public score와 validation score가 다를 때 어떤 점수를 더 믿어야 하는지 알고 싶다. 특히 제출 횟수가 제한된 대회에서는 이 판단이 최종 순위에 큰 영향을 준다.

## 18. 최종 정리

이번 개선 과정의 결론은 분명하다.

직접 CNN은 데이터 증강과 pseudo-label로 조금 좋아졌지만, 성능을 파격적으로 올린 핵심은 ResNet18 fine-tuning이었다. STL-10은 labeled train이 적고 unlabeled가 많은 구조라서, pretrained model과 pseudo-label 전략이 문제 특성에 잘 맞았다.

현재 최고 결과는 **0.9557**이고, 다음 제출에서 가장 가능성이 높은 개선은 **ResNet50 + EfficientNet-B0 + ResNet18 soft ensemble + TTA**이다. 목표 점수는 1차 0.9600 이상, 2차 0.9650 이상으로 잡는다.